In [1]:
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
from rasterstats import zonal_stats

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land"
)

heatday_dir = ROOT / "data" / "monthly_heatday_scores"
output_dir = ROOT / "data" / "variables"

geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/Heat-odisha/Maps/od_ids-drr_shapefiles/odisha_block_final_reduced.geojson"
)

output_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD GEOJSON
# =========================================================

gdf = gpd.read_file(geojson_path)

if "object_id" not in gdf.columns:
    raise ValueError("object_id column missing")

gdf = gdf[["object_id", "geometry"]]

# =========================================================
# MASTER STORAGE
# =========================================================

all_records = []

# =========================================================
# LOOP
# =========================================================

years = [2023, 2024]

for year in years:

    print(f"\n================ YEAR {year} ================")

    for month in range(1, 13):

        timeperiod = f"{year}_{month:02d}"

        raster_path = heatday_dir / f"HEATDAY_{year}_{month:02d}.tif"

        print(f"\nProcessing {timeperiod}")

        if not raster_path.exists():
            print("SKIP missing raster")
            continue

        # =================================================
        # ZONAL STATS
        # =================================================

        zs = zonal_stats(
            vectors=gdf,
            raster=str(raster_path),
            stats=["mean"],
            geojson_out=True,
            nodata=np.nan
        )

        # =================================================
        # STORE RESULTS
        # =================================================

        for f in zs:

            record = {
                "object_id": f["properties"]["object_id"],
                "mean_heatday": f["properties"]["mean"],
                "timeperiod": timeperiod
            }

            all_records.append(record)

        # =================================================
        # ALSO WRITE MONTHLY FILE
        # =================================================

        df_month = pd.DataFrame([
            {
                "object_id": f["properties"]["object_id"],
                "mean_heatday": f["properties"]["mean"],
                "timeperiod": timeperiod
            }
            for f in zs
        ])

        out_csv = output_dir / f"HEATDAY_{year}_{month:02d}.csv"
        df_month.to_csv(out_csv, index=False)

        print("Saved:", out_csv)

# =========================================================
# FINAL MASTER CSV
# =========================================================

df_all = pd.DataFrame(all_records)

final_csv = output_dir / "heatdays.csv"
df_all.to_csv(final_csv, index=False)

print("\n================ DONE ================")
print("MASTER FILE SAVED:", final_csv)


================ YEAR 2023 ================

Processing 2023_01
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_01.csv

Processing 2023_02
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_02.csv

Processing 2023_03
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_03.csv

Processing 2023_04
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_04.csv

Processing 2023_05
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_05.csv

Processing 2023_06
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_06.csv

Processing 2023_07
Saved: /home/root_1/Documents/CDL/repos/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_07.csv

Processing 2023_08
Saved: /hom